# Step 1: Prepare the GSDiff Model Environment
This cell sets up the required libraries, clones the original GSDiff repository, and downloads the required pretrained model weights (exactly as done in the original notebook).

In [1]:
!pip install -q gdown networkx matplotlib pyvis Shapely Pillow

%cd /content
!git clone https://github.com/SizheHu/GSDiff.git
%cd GSDiff

!mkdir -p outputs
%cd outputs

!gdown "1pk7SmvLZ8ON3OUL3SNxPRu73ndVKru0z"
!gdown "1tExX8LdrFpJfBQH5y2emC6BltBwf9tHx"

!tar -xvf *.tar*
!unzip -q topo-params.zip
%cd /content

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 17.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 73.5 MB/s eta 0:00:00
/content
Cloning into 'GSDiff'...
remote: Enumerating objects: 541, done.
remote: Counting objects: 100% (124/124), done.
remote: Compressing objects: 100% (106/106), done.
remote: Total 541 (delta 76), reused 19 (delta 18), pack-reused 417 (from 2)
Receiving objects: 100% (541/541), 926.95 KiB | 3.69 MiB/s, done.
Resolving deltas: 100% (333/333), done.
/content/GSDiff
/content/GSDiff/outputs
Downloading...
From (original): https://drive.google.com/uc?id=1pk7SmvLZ8ON3OUL3SNxPRu73ndVKru0z
From (redirected): https://drive.google.com/uc?id=1pk7SmvLZ8ON3OUL3SNxPRu73ndVKru0z&confirm=t&uuid=f412cfc0-e52d-44ad-a4d8-fc74c8ebd80f
To: /content/GSDiff/outputs/topo-params.zip
100% 513M/513M [00:10<00:00, 50.5MB/s] 
Traceback (most recent call last):
  File "/usr/local/bin/gdown", line 10, in <module>
    sys.exit(main())
             ^^^

# Step 2: Download Your Custom Floorplan Generator Repository
Replace the placeholder `<YOUR_GITHUB_REPO_URL>` with your actual GitHub repository URL after you publish it. This will pull your Python structured engine.

In [ ]:
%cd /content
!git clone "https://github.com/joeMo942/floor_plan_genration_with_doors/"

# CHANGE THIS to the name of your cloned repository folder!
REPO_FOLDER_NAME = "floor_plan_genration_with_doors"
%cd {REPO_FOLDER_NAME}

# Install the door placement engine dependencies
!pip install -r door_placement/requirements.txt

/content
fatal: destination path 'floor_plan_genration_with_doors' already exists and is not an empty directory.
/content/floor_plan_genration_with_doors


: 

# Step 3: Run the Pipeline
You can run your interactive python script right here in Colab. It will ask for your inputs directly underneath the cell!

In [4]:
!python run_full_pipeline.py

     AI FLOOR PLAN GENERATION & DOOR PLACEMENT PIPELINE

[STEP 1] Please enter the number of rooms for your layout:
  - Living Rooms    [default: 1]: Traceback (most recent call last):
  File "/content/floor_plan_genration_with_doors/run_full_pipeline.py", line 101, in <module>
    main()
  File "/content/floor_plan_genration_with_doors/run_full_pipeline.py", line 24, in main
    living_rooms    = get_input("  - Living Rooms   ", 1)
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/floor_plan_genration_with_doors/run_full_pipeline.py", line 21, in get_input
    val = input(f"{prompt} [default: {default_val}]: ").strip()
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt
^C


# Alternative: Run Programmatically (Without CLI Prompt)
If you'd prefer to script it directly in Python without answering the interactive console prompts, you can do it like this:

In [ ]:
import os
import sys
sys.path.append(os.path.abspath('.'))

from floorplan_generation.topology import generate_topology_from_form, visualize_agent_output
from floorplan_generation.inference import run_gsdiff_inference
from door_placement.pipeline import run_pipeline

user_input = {
    "property_type": "apartment",
    "rooms": {
        "living_rooms": 1,
        "bedrooms": 2,
        "master_bedrooms": 1,
        "bathrooms": 2,
        "kitchens": 1,
        "storage": 0
    }
}

# 1. Generate Topology
agent_nodes, agent_edges = generate_topology_from_form(user_input)
os.makedirs("outputs/custom_test", exist_ok=True)
visualize_agent_output(agent_nodes, agent_edges, save_path="outputs/custom_test/bubble_diagram.png")

# 2. Run GSDiff generation
json_dir = "outputs/custom_jsons"
run_gsdiff_inference(
    agent_nodes, agent_edges, 
    output_dir="outputs/custom_test", 
    json_dir=json_dir, 
    num_samples=15
)

# 3. Run Door Placement on specific plans (e.g. plan 0 and plan 5)
for plan_num in [0, 5]:
    input_json = os.path.join(json_dir, f"custom_pred_{plan_num}.json")
    out_path = f"outputs/final_door_placements/plan_{plan_num}"
    
    if os.path.exists(input_json):
        print(f"\nProcessing {input_json}...")
        run_pipeline(input_json=input_json, output_dir=out_path)


# Step 4: Run the Web UI Studio
Run this cell to start the modern React interface and expose it via a public ngrok link.

In [ ]:
import os
import time
from getpass import getpass
from pyngrok import ngrok

# 1. Install required packages
!pip install -q fastapi uvicorn pyngrok pydantic python-multipart

# 2. Build the React UI
print('Building React UI... (this takes a few seconds)')
!cd web_ui && npm install && npm run build

# 3. Ask for Ngrok Token and Start Server
NGROK_TOKEN = getpass('Enter your Ngrok authtoken (get it from https://dashboard.ngrok.com/get-started/your-authtoken): ')
!ngrok config add-authtoken {NGROK_TOKEN}

print('Starting FastAPI server...')
os.system('nohup uvicorn api.server:app --port 8000 & ')
time.sleep(3) # Wait for server to start

# 4. Open Ngrok Tunnel
public_url = ngrok.connect(8000)
print('\n' + '='*50)
print(f'🌟 STUDIO IS LIVE AT: {public_url.public_url}')
print('='*50 + '\n')